In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
!pip install -q dagshub mlflow
import mlflow
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret("DAGSHUB_TOKEN")
dagshub_username = user_secrets.get_secret("DAGSHUB_USERNAME")

os.environ['MLFLOW_TRACKING_USERNAME'] = dagshub_username
os.environ['MLFLOW_TRACKING_PASSWORD'] = dagshub_token

mlflow.set_tracking_uri(f"https://dagshub.com/{dagshub_username}/ML_assignment2.mlflow")
mlflow.end_run()

print(f"Successfully connected to {dagshub_username}'s MLflow server!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5

## Data Loading & Cleaning

In [3]:
import gc

train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

for col in train_transaction.select_dtypes('float64').columns:
    train_transaction[col] = train_transaction[col].astype('float32')

for col in train_identity.select_dtypes('float64').columns:
    train_identity[col] = train_identity[col].astype('float32')

train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')

del train_transaction, train_identity
gc.collect()

print(f"Setup Complete. Final Dataset Shape: {train.shape}")
print(f"Memory Usage: {train.memory_usage().sum() / 1024**2:.2f} MB")

mlflow.end_run()
mlflow.set_experiment("RandomForest_Training")

with mlflow.start_run(run_name="RF_Cleaning"):
    null_percent = train.isnull().sum() / len(train)
    cols_to_drop = null_percent[null_percent > 0.90].index.tolist()
    train.drop(columns=cols_to_drop, inplace=True)

    mlflow.log_param("missing_threshold", 0.90)
    mlflow.log_metric("cols_dropped", len(cols_to_drop))
    print(f"Cleaning complete. Dropped {len(cols_to_drop)} columns.")

Setup Complete. Final Dataset Shape: (590540, 434)
Memory Usage: 1056.53 MB
Cleaning complete. Dropped 12 columns.
🏃 View run RF_Cleaning at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1/runs/740ce4b11dda49f5832625ba1b2d848f
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1


## Feature Engineering

In [4]:
mlflow.end_run()
with mlflow.start_run(run_name="RF_Feature_Engineering"):
    train['Transaction_hour'] = (train['TransactionDT'] // 3600) % 24
    train['Transaction_day'] = (train['TransactionDT'] // (3600 * 24)) % 7

    train['card1_amt_mean'] = train.groupby('card1')['TransactionAmt'].transform('mean')
    train['amt_to_mean_card1'] = train['TransactionAmt'] / train['card1_amt_mean']

    train.drop(columns=['card1_amt_mean'], inplace=True)

    mlflow.log_param("added_features", ["Transaction_hour", "Transaction_day", "amt_to_mean_card1"])
    mlflow.log_metric("total_features_after_eng", len(train.columns))
    print(f"Feature engineering complete. Total columns: {len(train.columns)}")

Feature engineering complete. Total columns: 425
🏃 View run RF_Feature_Engineering at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1/runs/f35580878abc42ce912436cf871a2372
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1


## Feature Selection

In [5]:
from sklearn.model_selection import train_test_split

mlflow.end_run()
with mlflow.start_run(run_name="RF_Feature_Selection"):
    target = 'isFraud'
    drop_cols = [target, 'TransactionID', 'TransactionDT']
    constant_cols = [col for col in train.columns if train[col].nunique() <= 1]
    potential_drops = list(set(drop_cols + constant_cols))
    all_to_drop = [c for c in potential_drops if c in train.columns]
    
    X = train.drop(columns=all_to_drop)
    y = train[target]
    
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
    
    mlflow.log_param("dropped_identifiers", str(drop_cols))
    mlflow.log_param("constant_cols_dropped", str(constant_cols))
    mlflow.log_param("num_cat_features", len(cat_cols))
    mlflow.log_param("num_numeric_features", len(num_cols))
    mlflow.log_metric("final_feature_count", X.shape[1])
    print(f"Categorical features: {len(cat_cols)}")
    print(f"Numerical features: {len(num_cols)}")
    print(f"Final feature count: {X.shape[1]}")

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Categorical features: 29
Numerical features: 393
Final feature count: 422
🏃 View run RF_Feature_Selection at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1/runs/96e5d947595d4629804b1a37fed5743f
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1


## Training and Model Pipeline

In [6]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from imblearn.ensemble import BalancedRandomForestClassifier 

def build_preprocessor(num_cols, cat_cols):
    num_transformer = SimpleImputer(strategy='median')
    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ])
    return ColumnTransformer(transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

experiments = [
    {
        "run_name": "RF_baseline",
        "model": RandomForestClassifier(
            n_estimators=100, max_depth=12,
            n_jobs=-1, random_state=42, verbose=1
        )
    },
    {
        "run_name": "RF_deeper",
        "model": RandomForestClassifier(
            n_estimators=200,
            max_depth=20,
            min_samples_leaf=5,
            class_weight='balanced',
            n_jobs=-1,
            random_state=42,
            verbose=1
        )
    },
    {
        "run_name": "ExtraTrees",
        "model": ExtraTreesClassifier(
            n_estimators=100, max_depth=12,
            n_jobs=-1, random_state=42, verbose=1
        )
    },
]

for exp in experiments:
    mlflow.end_run()
    with mlflow.start_run(run_name=exp["run_name"]):
        clf = Pipeline(steps=[
            ('preprocessor', build_preprocessor(num_cols, cat_cols)),
            ('model', exp["model"])
        ])

        clf.fit(X_train, y_train)
        val_preds = clf.predict_proba(X_val)[:, 1]
        auc_score = roc_auc_score(y_val, val_preds)

        mlflow.log_params(exp["model"].get_params())
        mlflow.log_metric("val_auc", auc_score)
        mlflow.sklearn.log_model(clf, "model", registered_model_name=exp["run_name"])

        print(f"[{exp['run_name']}] Validation AUC: {auc_score:.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   28.2s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  1.1min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.3s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.6s finished
2026/05/01 17:57:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 17:57:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'RF_baseline' already exists. Creating a new versi

[RF_baseline] Validation AUC: 0.8760
🏃 View run RF_baseline at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1/runs/932241f298e340bfbfda495bcc65b34b
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   36.3s
[Parallel(n_jobs=-1)]: Done 192 tasks      | elapsed:  2.6min
[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed:  2.7min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.4s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:    1.9s
[Parallel(n_jobs=4)]: Done 200 out of 200 | elapsed:    2.0s finished
2026/05/01 18:00:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 18:00:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see:

[RF_deeper] Validation AUC: 0.9268
🏃 View run RF_deeper at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1/runs/d9c06a6b9d5342f79c047ad5e0395818
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   29.4s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  1.1min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.3s
[Parallel(n_jobs=4)]: Done 100 out of 100 | elapsed:    0.6s finished
2026/05/01 18:02:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 18:02:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'ExtraTrees' already exists. Creating a new versio

[ExtraTrees] Validation AUC: 0.8500
🏃 View run ExtraTrees at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1/runs/133a809c830a4a91980602b411e06086
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1


In [7]:
mlflow.end_run()
with mlflow.start_run(run_name="RF_deeper_300trees"):
    clf = Pipeline(steps=[
        ('preprocessor', build_preprocessor(num_cols, cat_cols)),
        ('model', RandomForestClassifier(
            n_estimators=300,
            max_depth=20,
            min_samples_leaf=5,
            class_weight='balanced',
            n_jobs=-1,
            random_state=42,
            verbose=1
        ))
    ])

    clf.fit(X_train, y_train)
    val_preds = clf.predict_proba(X_val)[:, 1]
    auc_score = roc_auc_score(y_val, val_preds)

    mlflow.log_params(clf.named_steps['model'].get_params())
    mlflow.log_metric("val_auc", auc_score)
    mlflow.sklearn.log_model(clf, "model", registered_model_name="RF_deeper_300trees")

    print(f"[RF_deeper_300trees] Validation AUC: {auc_score:.4f}")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   36.3s
[Parallel(n_jobs=-1)]: Done 192 tasks      | elapsed:  2.6min
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:  4.1min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.5s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:    1.9s
[Parallel(n_jobs=4)]: Done 300 out of 300 | elapsed:    3.0s finished
2026/05/01 18:07:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 18:07:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see:

[RF_deeper_300trees] Validation AUC: 0.9273
🏃 View run RF_deeper_300trees at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1/runs/022818bce36047b4afba3407774b2b91
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1


### Hyperparameter Tuning

In [8]:
from sklearn.model_selection import RandomizedSearchCV
import numpy as np

X_tune, _, y_tune, _ = train_test_split(X_train, y_train, train_size=0.10, stratify=y_train, random_state=42)

mlflow.end_run()
with mlflow.start_run(run_name="RF_hyperparameter_tuning"):
    
    preprocessor = build_preprocessor(num_cols, cat_cols)
    
    rf_model = RandomForestClassifier(
        class_weight='balanced',
        n_jobs=-1,
        random_state=42,
        verbose=0  
    )
    
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', rf_model)
    ])
    
    param_dist = {
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [12, 16, 20, 25],
        'model__min_samples_leaf': [1, 5, 10, 20],
        'model__max_features': ['sqrt', 'log2', 0.3]
    }
    
    search = RandomizedSearchCV(
    clf,
    param_distributions=param_dist,
    n_iter=5,
    scoring='roc_auc',
    cv=3,
    n_jobs=1,
    random_state=42,
    verbose=2
    )
    search.fit(X_tune, y_tune)
    
    best_model = search.best_estimator_
    val_preds = best_model.predict_proba(X_val)[:, 1]
    auc_score = roc_auc_score(y_val, val_preds)
    
    mlflow.log_params(search.best_params_)
    mlflow.log_metric("best_cv_auc", search.best_score_)
    mlflow.log_metric("val_auc", auc_score)
    mlflow.sklearn.log_model(best_model, "model", registered_model_name="RF_tuned")
    
    print(f"Best params: {search.best_params_}")
    print(f"Best CV AUC: {search.best_score_:.4f}")
    print(f"Validation AUC: {auc_score:.4f}")

Fitting 3 folds for each of 5 candidates, totalling 15 fits
[CV] END model__max_depth=25, model__max_features=sqrt, model__min_samples_leaf=20, model__n_estimators=100; total time=   6.5s
[CV] END model__max_depth=25, model__max_features=sqrt, model__min_samples_leaf=20, model__n_estimators=100; total time=   6.5s
[CV] END model__max_depth=25, model__max_features=sqrt, model__min_samples_leaf=20, model__n_estimators=100; total time=   6.5s
[CV] END model__max_depth=12, model__max_features=log2, model__min_samples_leaf=10, model__n_estimators=200; total time=   5.8s
[CV] END model__max_depth=12, model__max_features=log2, model__min_samples_leaf=10, model__n_estimators=200; total time=   5.8s
[CV] END model__max_depth=12, model__max_features=log2, model__min_samples_leaf=10, model__n_estimators=200; total time=   5.9s
[CV] END model__max_depth=20, model__max_features=sqrt, model__min_samples_leaf=20, model__n_estimators=200; total time=  10.3s
[CV] END model__max_depth=20, model__max_fea

2026/05/01 18:12:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 18:12:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'RF_tuned' already exists. Creating a new version of this model...
2026/05/01 18:12:22 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RF_tuned, version 2
Created version '2' of model 'RF_tuned'.


Best params: {'model__n_estimators': 100, 'model__min_samples_leaf': 20, 'model__max_features': 'sqrt', 'model__max_depth': 25}
Best CV AUC: 0.8778
Validation AUC: 0.8839
🏃 View run RF_hyperparameter_tuning at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1/runs/099d7a6f23664191879aef87b0bcfa0b
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1


In [9]:
mlflow.end_run()
with mlflow.start_run(run_name="RF_Final_Hybrid"):
    preprocessor = build_preprocessor(num_cols, cat_cols)
    final_rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=25,         
        min_samples_leaf=2,   
        max_features='sqrt',   
        class_weight='balanced',
        n_jobs=-1,
        random_state=42,
        verbose=1
    )

    clf_final = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', final_rf)
    ])

    print("Training the Hybrid Model on full data...")
    clf_final.fit(X_train, y_train)

    val_probs = clf_final.predict_proba(X_val)[:, 1]
    auc_score = roc_auc_score(y_val, val_probs)

    mlflow.log_params(final_rf.get_params())
    mlflow.log_metric("val_auc", auc_score)
    mlflow.sklearn.log_model(clf_final, "model", registered_model_name="RF_Final_Hybrid")

    print(f"Hybrid Model Complete! Validation AUC: {auc_score:.4f}")

Training the Hybrid Model on full data...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   39.8s
[Parallel(n_jobs=-1)]: Done 192 tasks      | elapsed:  2.9min
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:  4.6min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    0.5s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:    2.2s
[Parallel(n_jobs=4)]: Done 300 out of 300 | elapsed:    3.4s finished
2026/05/01 18:17:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/01 18:17:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see:

Hybrid Model Complete! Validation AUC: 0.9302
🏃 View run RF_Final_Hybrid at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1/runs/4bdac062426b4a04b508bc4d55a49211
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/1
